In [1]:
import os

from azure.ai.formrecognizer import FormTrainingClient, FormRecognizerClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")

BOARDING_PASS_RECOGNIZER_ENDPOINT = os.environ["BOARDING_PASS_RECOGNIZER_ENDPOINT"]
BOARDING_PASS_RECOGNIZER_KEY = os.environ["BOARDING_PASS_RECOGNIZER_KEY"]
BOARDING_PASS_TRAINING_URL = os.environ["BOARDING_PASS_TRAINING_URL"]

In [3]:
form_training_client = FormTrainingClient(endpoint=BOARDING_PASS_RECOGNIZER_ENDPOINT, credential=AzureKeyCredential(BOARDING_PASS_RECOGNIZER_KEY))

In [4]:
saved_model_list = form_training_client.list_custom_models()

In [5]:
training_process = form_training_client.begin_training(BOARDING_PASS_TRAINING_URL, use_training_labels=True)
custom_model = training_process.result()

## Getting Model Info

In [6]:
custom_model.model_id

'81186b69-1a9e-4239-8250-20587f483cab'

In [7]:
custom_model.status

'ready'

In [8]:
custom_model.training_documents

[TrainingDocumentInfo(name=boarding_pass_01_JAMES_MITCHELL.pdf, status=succeeded, page_count=1, errors=[], model_id=None),
 TrainingDocumentInfo(name=boarding_pass_02_SARAH_CHEN.pdf, status=succeeded, page_count=1, errors=[], model_id=None),
 TrainingDocumentInfo(name=boarding_pass_03_ROBERT_GARCIA.pdf, status=succeeded, page_count=1, errors=[], model_id=None),
 TrainingDocumentInfo(name=boarding_pass_04_EMILY_THOMPSON.pdf, status=succeeded, page_count=1, errors=[], model_id=None),
 TrainingDocumentInfo(name=boarding_pass_05_MICHAEL_JOHNSON.pdf, status=succeeded, page_count=1, errors=[], model_id=None),
 TrainingDocumentInfo(name=boarding_pass_06_JENNIFER_PATEL.pdf, status=succeeded, page_count=1, errors=[], model_id=None),
 TrainingDocumentInfo(name=boarding_pass_07_DAVID_WONG.pdf, status=succeeded, page_count=1, errors=[], model_id=None),
 TrainingDocumentInfo(name=boarding_pass_08_AMANDA_ROSS.pdf, status=succeeded, page_count=1, errors=[], model_id=None),
 TrainingDocumentInfo(name=

In [9]:
for doc in custom_model.training_documents:
    print("Document name: {}".format(doc.name))
    print("Document status: {}".format(doc.status))
    print("Document page count: {}".format(doc.page_count))
    print("Document errors: {}".format(doc.errors))

Document name: boarding_pass_01_JAMES_MITCHELL.pdf
Document status: succeeded
Document page count: 1
Document errors: []
Document name: boarding_pass_02_SARAH_CHEN.pdf
Document status: succeeded
Document page count: 1
Document errors: []
Document name: boarding_pass_03_ROBERT_GARCIA.pdf
Document status: succeeded
Document page count: 1
Document errors: []
Document name: boarding_pass_04_EMILY_THOMPSON.pdf
Document status: succeeded
Document page count: 1
Document errors: []
Document name: boarding_pass_05_MICHAEL_JOHNSON.pdf
Document status: succeeded
Document page count: 1
Document errors: []
Document name: boarding_pass_06_JENNIFER_PATEL.pdf
Document status: succeeded
Document page count: 1
Document errors: []
Document name: boarding_pass_07_DAVID_WONG.pdf
Document status: succeeded
Document page count: 1
Document errors: []
Document name: boarding_pass_08_AMANDA_ROSS.pdf
Document status: succeeded
Document page count: 1
Document errors: []
Document name: boarding_pass_09_CARLOS_RAMI

## Recognizing Custom Forms

In [10]:
form_recognizer_client = FormRecognizerClient(endpoint=BOARDING_PASS_RECOGNIZER_ENDPOINT, credential=AzureKeyCredential(BOARDING_PASS_RECOGNIZER_KEY))

In [11]:
local_image_path = r"../material_preparation_step/boarding_passes/boarding-avkash.pdf"

with open(local_image_path, "rb") as f:
    custom_test_action = form_recognizer_client.begin_recognize_custom_forms(model_id=custom_model.model_id, form=f)
    custom_test_action_result = custom_test_action.result()

In [12]:
for recognized_content in custom_test_action_result:
    print("Form type: {}".format(recognized_content.form_type))
    for name, field in recognized_content.fields.items():
        print("Field '{}' has label '{}' with value '{}' and a confidence score of {}".format(
            name,
            field.label_data.text if field.label_data else name,
            field.value,
            field.confidence
        ))

Form type: custom:81186b69-1a9e-4239-8250-20587f483cab
Field 'Gate' has label 'Gate' with value 'G1' and a confidence score of 0.989
Field 'Passenger Name' has label 'Passenger Name' with value 'Avkash Chauhan' and a confidence score of 0.99
Field 'Boarding Time' has label 'Boarding Time' with value '10:00 AM PST' and a confidence score of 0.981
Field 'To' has label 'To' with value 'Chicago' and a confidence score of 0.99
Field 'From' has label 'From' with value 'San Francisco' and a confidence score of 0.99
Field 'Date' has label 'Date' with value 'April 20, 2022' and a confidence score of 0.693
Field 'Flight No' has label 'Flight No' with value '234' and a confidence score of 0.99
Field 'Seat' has label 'Seat' with value '20A' and a confidence score of 0.99
Field 'Class' has label 'Class' with value 'E' and a confidence score of 0.99
Field 'Carrier' has label 'Carrier' with value 'UA' and a confidence score of 0.99
